# Architecture

```mermaid
flowchart LR
  R[Python repository] --> P[Parser Service]
  P --> N[cpg.nodes.v1]
  P --> E[cpg.edges.v1]
  P --> M[cpg.source-metadata.v1]
  P --> X[cpg.parser-errors.v1]
  N --> C[Kafka Connect]
  E --> C
  C --> G[Neo4j]
  C --> Q[cpg.neo4j-dlq.v1]
  M --> S[Spark Structured Streaming]
  S --> D[MongoDB]
  S --> K[(Checkpoint)]
  X --> L[Parser error evidence]
```

Graph topology goes directly from Kafka Connect to Neo4j. Spark is used only
for source metadata; parser failures and connector failures have separate paths.

## Approach and rationale

**Approach:** The pipeline separates graph topology, source metadata, parser
errors, and connector failures at the Kafka boundary. Kafka Connect owns the
node/edge branch, while one Spark Structured Streaming query owns only the
metadata branch and persists its checkpoint on a Docker volume.

**Why this approach:** The split follows the assignment's required sink paths
and gives each consumer the smallest possible contract. Sending topology
directly through Kafka Connect avoids an unnecessary Spark transformation and
keeps Neo4j ingestion independent from metadata analytics. Separate error paths
prevent a malformed source file or sink record from stopping valid traffic.

**Alternatives and trade-offs:** A single topic or a Spark job for both sinks
would reduce the number of components but couple unrelated schemas, recovery
semantics, and failure handling. One partition and one broker make ordering and
the classroom demonstration reproducible, at the cost of throughput and
production-grade availability.

In [1]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
result = subprocess.run(
    ['docker', 'compose', 'config', '--services'],
    cwd=root, capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
required = {'broker', 'connect', 'neo4j', 'mongo', 'spark-metadata'}
assert required <= set(result.stdout.splitlines())
print('PASS: all required architecture services are declared')

broker
kafka-init
mongo
neo4j
neo4j-init
spark-metadata
connect
connect-init
PASS: all required architecture services are declared


## Reflection

**Worked:** Kafka Connect and Spark operate on separate branches, so graph topology never passes through Spark.

**Failed:** Connector creation was asynchronous and an immediate status request could observe a temporary 404.

**Resolution:** `register-wait.sh` now retries until both the connector and its task are `RUNNING`. Single-node Kafka and replication factor one remain documented educational limits.